<a href="https://colab.research.google.com/github/AdisakJack/CMPE-259-Project-BALLPARK/blob/main/CMPE_259_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup

In [13]:
# Run once per session. If Colab asks you to restart the runtime after installing, restart and re-run from here.
!pip install -q -U google-genai
!pip install -q git+https://github.com/jldbc/pybaseball.git   # GitHub build, the version this project was developed with

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-

In [14]:
import re
import time

import pandas as pd
from google import genai
from google.genai import errors, types
from google.colab import userdata
from pybaseball import batting_stats_bref, bwar_bat, cache

cache.enable()   # cache Baseball Reference downloads locally (they are rate limited)

In [15]:
# ---- Configuration (change things here, nowhere else) ----
API_KEY_SECRET = "BALLPARK"          # name of the Colab secret that holds your Gemini API key
GEMINI_MODEL = "gemini-3.5-flash-lite"    # pinned model name; list available models in Appendix B

client = genai.Client(api_key=userdata.get(API_KEY_SECRET))

## 2. Data tools
Two functions that the model can call. Each downloads a table once per session (cached), filters it, and returns a small dictionary with the source attached. Errors are returned as data (`{"error": ...}`) so the model can say "I couldn't find that" instead of guessing.

In [16]:
_season_cache = {}
_war_df = None
STAT_COLS = ["Name", "Tm", "PA", "BA", "OBP", "SLG", "OPS", "HR", "RBI"]


def _clean(v):
    if pd.isna(v):
        return None
    return v.item() if hasattr(v, "item") else v


def get_batting_stats(player_name: str, season: int) -> dict:
    """Get a batter's Baseball Reference season stats (BA, OBP, SLG, OPS, HR, RBI, PA).

    Args:
        player_name: Full name, e.g. "Aaron Judge".
        season: Four-digit MLB season, e.g. 2025.
    """
    if season not in _season_cache:
        _season_cache[season] = batting_stats_bref(season)   # one download per season
    df = _season_cache[season]
    rows = df[df["Name"].str.contains(player_name, case=False, regex=False, na=False)]
    if rows.empty:
        return {"error": f"No batter named '{player_name}' found for {season}."}
    if rows["Name"].nunique() > 1:
        return {"error": f"Multiple players match '{player_name}': {sorted(rows['Name'].unique())}. Ask the user to clarify."}
    cols = [c for c in STAT_COLS if c in rows.columns]
    records = [{k: _clean(v) for k, v in r.items()} for _, r in rows[cols].iterrows()]
    out = {"season": season, "source": "Baseball Reference via pybaseball"}
    if len(records) == 1:
        out.update(records[0])
    else:
        out["note"] = ("Multiple rows match this name: either one player traded mid-season "
                       "or different players who share the name. Check Tm and PA; if unclear, ask the user.")
        out["rows"] = records
    return out


def get_bwar(player_name: str, season: int) -> dict:
    """Get a batter's Baseball Reference bWAR for a season.

    Args:
        player_name: Full name, e.g. "Mike Trout".
        season: Four-digit MLB season.
    """
    global _war_df
    if _war_df is None:
        _war_df = bwar_bat()
    rows = _war_df[(_war_df["name_common"].str.contains(player_name, case=False, regex=False, na=False))
                   & (_war_df["year_ID"] == season)]
    if rows.empty:
        return {"error": f"No bWAR found for '{player_name}' in {season}."}
    return {"player": rows["name_common"].iloc[0], "season": season,
            "bwar": float(rows["WAR"].sum()), "source": "Baseball Reference bWAR"}

In [21]:
import os
import inspect
from pybaseball import (playerid_lookup, statcast_batter_exitvelo_barrels,
                        statcast_batter_bat_tracking, statcast_batter_expected_stats)

DATA_DIR = "statcast_data"
os.makedirs(DATA_DIR, exist_ok=True)
_board_cache = {}
ID_COLUMNS = ["player_id", "id", "batter", "mlb_id", "mlbam"]


def _clean_value(v):
    if pd.isna(v):
        return None
    v = v.item() if hasattr(v, "item") else v
    return round(v, 3) if isinstance(v, float) else v


def _load_board(key, fn, year):
    """Load a Savant leaderboard with ALL players (not just qualified).
    Order: memory cache -> CSV on disk -> download and save."""
    if (key, year) in _board_cache:
        return _board_cache[(key, year)]
    path = f"{DATA_DIR}/{key}_{year}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
    else:
        # Leaderboards default to qualified players only; lower any 'min...' argument to 1
        kwargs = {p: 1 for p in inspect.signature(fn).parameters if p.lower().startswith("min")}
        df = fn(year, **kwargs)
        df.to_csv(path, index=False)
    _board_cache[(key, year)] = df
    return df


def _resolve_player(player_name, season):
    """Return (mlbam_id, None) or (None, error_dict)."""
    parts = player_name.strip().split()
    if len(parts) < 2:
        return None, {"error": f"Please give a first and last name (got '{player_name}')."}
    first, last = " ".join(parts[:-1]), parts[-1]
    ids = playerid_lookup(last, first)
    if ids is None or len(ids) == 0:
        return None, {"error": f"No player named '{player_name}' found in the player ID table."}
    active = ids[(ids["mlb_played_first"] <= season) & (ids["mlb_played_last"] >= season)]
    pool = active if len(active) > 0 else ids
    if len(pool) > 1:
        cands = [f"{r['name_first']} {r['name_last']} ({int(r['mlb_played_first'])}-{int(r['mlb_played_last'])}, id {int(r['key_mlbam'])})"
                 for _, r in pool.iterrows()]
        return None, {"error": f"Multiple players match '{player_name}': {cands}. Ask the user which one."}
    return int(pool["key_mlbam"].iloc[0]), None


def _lookup_statcast(player_name, season, key, fn, label):
    mlbam, err = _resolve_player(player_name, season)
    if err:
        return err
    try:
        df = _load_board(key, fn, season)
    except Exception as e:
        return {"error": f"Could not load {label} data for {season}: {type(e).__name__}: {e}. "
                         f"This data may not exist for that season."}
    id_col = next((c for c in ID_COLUMNS if c in df.columns), None)
    if id_col is None:
        return {"error": f"No player-id column found in the {label} data. Columns: {list(df.columns)}"}
    rows = df[pd.to_numeric(df[id_col], errors="coerce") == mlbam]
    if rows.empty:
        return {"error": f"No {label} data for '{player_name}' in {season} "
                         f"(player may not have enough plate appearances, or the season has no data)."}
    out = {k: _clean_value(v) for k, v in rows.iloc[0].items()}
    out.update({"player": player_name, "season": season,
                "source": "Baseball Savant (Statcast) via pybaseball"})
    return out


def get_barrel_stats(player_name: str, season: int) -> dict:
    """Get a batter's Statcast batted-ball quality for a season: barrel rate, exit velocity and hard-hit rate.

    Likely columns: brl_percent = barrels per batted ball event (%), brl_pa = barrels per plate
    appearance (%), avg_hit_speed = average exit velocity (mph), max_hit_speed = maximum exit
    velocity (mph), ev95percent = hard-hit rate (%), avg_hit_angle (degrees), attempts = number of
    batted ball events (the sample size).

    Args:
        player_name: Full name, e.g. "Aaron Judge".
        season: Four-digit MLB season, e.g. 2025.
    """
    return _lookup_statcast(player_name, season, "batter_barrels",
                            statcast_batter_exitvelo_barrels, "barrel/exit-velocity")


def get_bat_tracking(player_name: str, season: int) -> dict:
    """Get a batter's Statcast bat-tracking data for a season (swing speed). Typically includes
    average bat speed (mph), swing length, and rates such as hard-swing, squared-up and blast
    rates. Bat tracking only exists for recent seasons; an error is returned for earlier ones.

    Args:
        player_name: Full name, e.g. "Aaron Judge".
        season: Four-digit MLB season, e.g. 2025.
    """
    return _lookup_statcast(player_name, season, "batter_bat_track",
                            statcast_batter_bat_tracking, "bat-tracking")


def get_expected_stats(player_name: str, season: int) -> dict:
    """Get a batter's Statcast expected stats for a season: xBA, xSLG, xwOBA next to the actual
    BA, SLG and wOBA (est_ba, est_slg, est_woba vs ba, slg, woba), plus pa and bip sample sizes.
    Useful for asking who over- or under-performed their batted-ball quality.

    Args:
        player_name: Full name, e.g. "Aaron Judge".
        season: Four-digit MLB season, e.g. 2025.
    """
    return _lookup_statcast(player_name, season, "batter_expected",
                            statcast_batter_expected_stats, "expected-stats")

## 3. Assistant
`ask(question, use_tools=True)` returns `(answer, calls, results)`:
- `answer`: the model's text,
- `calls`: the tool calls it made (empty means it answered from memory),
- `results`: what the tools returned (used by the evaluation's grounding check).

`use_tools=False` is the no-tools baseline: same model, no tools, and a prompt that allows answering from memory.

In [22]:
SYSTEM_TOOLS = """You are BALLPARK, an MLB stats assistant.
- Use the provided tools for every statistic. Never state a number from memory.
- If a tool returns an error or no data, say you couldn't find it. Do not guess.
- Cite the source and season for each stat.
- If a question is ambiguous (e.g. "who is better?"), say which stats you are comparing.
- If a question is outside MLB player stats, say so and don't answer it."""

SYSTEM_NOTOOLS = """You are BALLPARK, an MLB stats assistant.
Answer from your own knowledge. If you are unsure of a number, say so."""

TOOLS = [get_batting_stats, get_bwar, get_barrel_stats, get_bat_tracking, get_expected_stats]

def _generate(model, question, config, max_tries=5):
    """Retry on rate limits (429) and overloads (503)."""
    for attempt in range(max_tries):
        try:
            return client.models.generate_content(model=model, contents=question, config=config)
        except (errors.ServerError, errors.ClientError) as e:
            code = getattr(e, "code", None)
            if code in (429, 503) and attempt < max_tries - 1:
                time.sleep(5 * 2 ** attempt)
            else:
                raise


def ask(question, use_tools=True, model=GEMINI_MODEL, temperature=0.0):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_TOOLS if use_tools else SYSTEM_NOTOOLS,
        temperature=temperature,
        tools=TOOLS if use_tools else None,
    )
    resp = _generate(model, question, config)

    calls, results = [], []
    for content in (resp.automatic_function_calling_history or []):
        for part in (content.parts or []):
            if part.function_call:
                calls.append((part.function_call.name, dict(part.function_call.args)))
            if part.function_response:
                results.append(part.function_response.response)
    return (resp.text or ""), calls, results

## 4. Demo

In [24]:
question = "What was Willie Mays bating average in 1965?"

answer, calls, results = ask(question, use_tools=True)
print("WITH TOOLS\n", answer)
print("\ntool calls:", calls)       # should list two get_batting_stats calls
print("tool results:", results)

answer_nt, calls_nt, _ = ask(question, use_tools=False)
print("\nNO TOOLS\n", answer_nt)
print("\ntool calls:", calls_nt)   # should be empty

WITH TOOLS
 I'm sorry, but my data for Baseball Reference batting statistics only covers the 2008 season onward, so I cannot retrieve Willie Mays' batting average from 1965.

tool calls: [('get_batting_stats', {'player_name': 'Willie Mays', 'season': 1965})]
tool results: [{'error': "Failed to invoke function get_batting_stats with converted arguments {'player_name': 'Willie Mays', 'season': 1965} from model returned function call argument {'player_name': 'Willie Mays', 'season': 1965} because of error Year must be 2008 or later"}]

NO TOOLS
 In 1965, Willie Mays had a batting average of **.317**. 

That was an iconic season for him—he won his second National League MVP award after hitting a career-high 52 home runs and leading the Giants to a thrilling pennant race.

tool calls: []


## 5. Evaluation
**Workflow**
1. Define the evaluation functions (next cell).
2. Smoke-test on three queries.
3. Run the full query set (remove `only_ids`). Raw responses are saved after every query.
4. Download `results_raw.csv`, fill the manual columns (`correct`, `appropriate`, `failure_type`, `notes`) in a spreadsheet, save it as `results_scored.csv`, and upload it.
5. Run `summarize()` for the tables used in the README and slides.

Look up the `expected_numbers` in `ballpark_test_queries.csv` yourself on Baseball Reference *before* running, so the check is independent of the assistant.

In [19]:
NUM_RE = re.compile(r"-?\d*\.\d+|-?\d+")

# Use this fixed label set for the failure_type column so failures can be counted.
FAILURE_TYPES = [
    "hallucinated_stat", "wrong_tool", "no_tool_call", "name_mismatch",
    "data_missing", "ignored_ambiguity", "overconfident", "tool_error",
    "rate_limit", "followed_injection", "leaked_prompt", "none",
]


def _numbers(text):
    return [float(x) for x in NUM_RE.findall(str(text))]


def _expected_numbers(cell):
    """Parse expected_numbers ('0.289;0.311'). Ignore unfilled LOOKUP cells."""
    if pd.isna(cell) or "LOOKUP" in str(cell) or not str(cell).strip():
        return []
    return [float(x) for x in str(cell).split(";") if x.strip()]


def auto_numeric_match(answer, expected, tol=0.0006):
    """True if every expected number appears (within tol) among the numbers in the answer."""
    if not expected:
        return None
    found = _numbers(answer)
    return all(any(abs(f - e) <= tol for f in found) for e in expected)


def unsupported_numbers(answer, tool_results):
    """Numbers in the answer that never appear in the tool output (best-effort flag).
    Years and small integers are ignored."""
    if tool_results in ("", None):
        return None
    allowed = _numbers(tool_results)
    flagged = []
    for n in _numbers(answer):
        if 1900 <= n <= 2100 or (float(n).is_integer() and abs(n) <= 10):
            continue
        if not any(abs(n - a) <= 0.0006 for a in allowed):
            flagged.append(n)
    return flagged


def run_eval(queries_csv, configs, out_csv="results_raw.csv", delay=3, only_ids=None):
    """Run every query through every config; save raw results after every query."""
    queries = pd.read_csv(queries_csv)
    if only_ids is not None:
        queries = queries[queries["id"].isin(only_ids)]
    rows = []
    for _, q in queries.iterrows():
        for cfg_name, fn in configs.items():
            t0 = time.time()
            error = ""
            try:
                out = fn(q["query"])
                answer, calls = out[0], out[1]
                tool_results = out[2] if len(out) > 2 else ""
            except Exception as e:
                answer, calls, tool_results, error = "", [], "", repr(e)
            expected = _expected_numbers(q["expected_numbers"])
            rows.append({
                "id": q["id"],
                "set": q["set"],
                "category": q["category"],
                "config": cfg_name,
                "query": q["query"],
                "expected_behavior": q["expected_behavior"],
                "expected_numbers": q["expected_numbers"],
                "response": answer,
                "tools_called": str(calls),
                "tool_called": len(calls) > 0,
                "latency_s": round(time.time() - t0, 2),
                "auto_numeric_match": auto_numeric_match(answer, expected),
                "unsupported_numbers": str(unsupported_numbers(answer, tool_results)),
                "error": error,
                # ---- fill these by hand ----
                "correct": "",        # 1 = correct, 0.5 = partly, 0 = wrong
                "appropriate": "",    # 1/0: did it do the expected behavior
                "failure_type": "",   # one label from FAILURE_TYPES
                "notes": "",
            })
            time.sleep(delay)  # free-tier rate limits
        pd.DataFrame(rows).to_csv(out_csv, index=False)
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df


def summarize(scored_csv="results_scored.csv"):
    """Summary tables for the README / slides. Run after filling the manual columns."""
    df = pd.read_csv(scored_csv, dtype={"failure_type": str, "notes": str})
    for col in ["correct", "appropriate", "latency_s"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    core = df[df["set"] == "core"]
    safety = df[df["set"] == "safety"]

    print("=== Core query set: mean score by configuration ===")
    print(core.groupby("config")[["correct", "appropriate", "latency_s"]].mean().round(3))

    print("\n=== Core query set: mean 'correct' by category x configuration ===")
    print(core.pivot_table(index="category", columns="config", values="correct", aggfunc="mean").round(2))

    print("\n=== Tool usage rate by configuration ===")
    print(core.groupby("config")["tool_called"].mean().round(2))

    print("\n=== Failure types by configuration ===")
    fails = df[df["failure_type"].notna() & (df["failure_type"] != "none") & (df["failure_type"] != "")]
    print(fails.groupby(["config", "failure_type"]).size().unstack(fill_value=0))

    print("\n=== Safety / robustness tests ===")
    print(safety[["id", "category", "config", "appropriate", "failure_type", "notes"]].to_string(index=False))
    return df

In [20]:
# Each configuration is a function: query text -> (answer, calls, tool_results)
configs = {
    "gemini_tools":   lambda q: ask(q, use_tools=True),
    "gemini_notools": lambda q: ask(q, use_tools=False),
}

# Smoke test on three queries first
test_df = run_eval(
    "ballpark_test_queries.csv",
    configs,
    out_csv="results_smoke_test.csv",
    delay=3,
    only_ids=["Q01", "Q05", "S02"],
)
test_df[["id", "config", "response", "tools_called", "auto_numeric_match", "error"]]

FileNotFoundError: [Errno 2] No such file or directory: 'ballpark_test_queries.csv'

In [ ]:
# Full run (about 32 queries x 2 configurations; takes a while because of rate-limit delays)
results_df = run_eval(
    "ballpark_test_queries.csv",
    configs,
    out_csv="results_raw.csv",
    delay=3,
)
results_df[["id", "config", "tool_called", "auto_numeric_match", "unsupported_numbers", "error"]].head(20)

In [ ]:
# After scoring by hand and uploading results_scored.csv:
scored_df = summarize("results_scored.csv")